In [1]:
import os
import torch
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.vectorstores import Chroma
from langchain_classic.retrievers import ParentDocumentRetriever, EnsembleRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.retrievers import BM25Retriever
from langchain_core.stores import InMemoryByteStore
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
import gradio as gr
import gc

W0616 15:18:53.275000 22504 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [2]:
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU detected")
else:
    print("No GPU")

GPU detected


In [3]:
docs_path = "C:/Users/Rizky/OneDrive/Documents/Python/finetuning/Knowledge_documents_RAG"
loader =DirectoryLoader(
    docs_path,
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader
)

all_docs = loader.load()
print(f"load berhasil")

for doc in all_docs:
    # Contoh ekstraksi tahun dari isi teks atau path dokumen
    if "2021" in doc.page_content:
        doc.metadata["year"] = "2021"
    elif "2023" in doc.page_content:
        doc.metadata["year"] = "2023"
    else:
        doc.metadata["year"] = "unknown"

print("Pengayaan Metadata selesai")

load berhasil
Pengayaan Metadata selesai


In [4]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2500, chunk_overlap=250)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

In [5]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device":"cuda"}
)

In [6]:
vector_store = Chroma(
    collection_name="split_parents",
    embedding_function=embedding_model
)

C:\Users\Rizky\AppData\Local\Temp\ipykernel_22504\3445327890.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


In [7]:
storage = InMemoryByteStore()

In [8]:
retriever = ParentDocumentRetriever(
    vectorstore = vector_store,
    docstore = storage,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_type="similarity",
    search_kwargs={"k": 15}
)

batch_size = 50 

for i in range(0, len(all_docs), batch_size):
    batch_docs = all_docs[i : i + batch_size]
    retriever.add_documents(batch_docs)
    print(f"Berhasil memproses dokumen {i} sampai {i + len(batch_docs)} dari total {len(all_docs)}")

print("Semua dokumen berhasil dimasukkan ke ChromaDB!")

Berhasil memproses dokumen 0 sampai 50 dari total 1949
Berhasil memproses dokumen 50 sampai 100 dari total 1949
Berhasil memproses dokumen 100 sampai 150 dari total 1949
Berhasil memproses dokumen 150 sampai 200 dari total 1949
Berhasil memproses dokumen 200 sampai 250 dari total 1949
Berhasil memproses dokumen 250 sampai 300 dari total 1949
Berhasil memproses dokumen 300 sampai 350 dari total 1949
Berhasil memproses dokumen 350 sampai 400 dari total 1949
Berhasil memproses dokumen 400 sampai 450 dari total 1949
Berhasil memproses dokumen 450 sampai 500 dari total 1949
Berhasil memproses dokumen 500 sampai 550 dari total 1949
Berhasil memproses dokumen 550 sampai 600 dari total 1949
Berhasil memproses dokumen 600 sampai 650 dari total 1949
Berhasil memproses dokumen 650 sampai 700 dari total 1949
Berhasil memproses dokumen 700 sampai 750 dari total 1949
Berhasil memproses dokumen 750 sampai 800 dari total 1949
Berhasil memproses dokumen 800 sampai 850 dari total 1949
Berhasil memproses

In [9]:
query = "Apa yang dimaksud dengan kondisi ketika nilai upah minimum melebihi rata-rata konsumsi rumah tangga?"
relevant_docs = retriever.invoke(query)
 
for i, doc in enumerate(relevant_docs, start=1):
    print(f"--- Dokumen {i} ---")
    print(doc.page_content)
    print()

--- Dokumen 1 ---
PRESIDEN
REPUBLIK INDONESIA
-9-
Yang dimaksud dengan "Median Upah Kabf Kota"
adalah rata-rata median Upah Pekerja/Burrrh di
luar penyelenggara negara 3 (tiga) tahun terakhir
pada kabupaten/ kota yang bersangkutan.
Yang dimaksud dengan "Median Upah Provinsi"
adalah rata-rata median Upah Pekerja/Buruh di
luar penyelenggara negara 3 (tiga) tahun terakhir
pada provinsi yang bersangkutan.
Yang dimaksud dengan "UMP1I1" adalah Upah
minimum provinsi tahun berjalan.
Hurrrf d
Yang dimaksud dengan "UMK1t+ty" adalah nilai
Upah minimum kabupaten/kota yang akan
ditetapkan.
Yang dimaksud dengan "UMKlRt;" adalah nilai
Upah minimum kabupaten/kota dengan
mempertimbangkan faktor paritas daya beli.
Yang dimaksud dengan "UMKlRz;" adalah nilai
Upah minimum kabupaten/kota dengan
mempertimbangkan faktor tingkat penyerapan
tenaga kerja.
Yang dimaksud dengan "UMK1tr"s;" adalah nilai
Upah minimum kabupatenlkota dengan
mempertimbangkan faktor median Upah
Pekerja/Buruh di luar penyelenggara negar

In [10]:
bm25_retriever = BM25Retriever.from_documents(all_docs)
bm25_retriever.k = 10

In [11]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, retriever],
    weights=[0.5, 0.5]
)

In [12]:
new_query = "Kapan upah minimum kabupaten/kota berlaku efektif?"
hybrid_relevant_docs = hybrid_retriever.invoke(new_query)

for i, doc in enumerate(hybrid_relevant_docs, start=1):
    print(f"---Dokumen {i}---")
    print(doc.page_content)
    print()

---Dokumen 1---
8
PRESIDEN
REPUBLIK INDONESIA
-7 -
b. penetapan Upah minimum provinsi pertama kali
dilakukan oleh gubernur atau penjabat gubernur
paling lambat tanggal 21 November tahun
berjalan; dan
c. penyesuaian Upah minimum provinsi pertama
kali dilakukan oleh gubernur atau penjabat
gubernur paling lambat tanggal 2l November
tahun berikutnya.
(2) Penetapan Upah minimum provinsi pertama kali
sebagaimana dimaksud pada ayat (1) huruf b,
sebesar nilai Upah minimum provinsi induk.
Ketentuan ayat (21 Pasal 29 diubah sehingga berbunyi
sebagai berikut:
Pasal 29
(1) Upah minimum provinsi ditetapkan dengan
Keputusan Gubernur dan diumumkan paling lambat
tanggal 21 November tahun berjalan.
(21 Dalam hal tanggal 2l November jatuh pada
hari Minggu, hari libur nasional, atau hari libur
resmi, Upah minimum provinsi ditetapkan dan
diumumkan oleh gubernur atau penjabat gubernur
1 (satu) hari sebelum hari Minggu, hari libur nasional,
atau hari libur resmi.
(3) Upah minimum provinsi sebagaimana dimaks

In [13]:
rerank_model = HuggingFaceCrossEncoder(model_name='BAAI/bge-reranker-large')
compressor = CrossEncoderReranker(model=rerank_model, top_n=2)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=hybrid_retriever
)

In [14]:
final_docs = compression_retriever.invoke(query)
 
for i, doc in enumerate(final_docs, start=1):
    print(f"--- Dokumen {i} ---")
    print(doc.page_content)
    print()

--- Dokumen 1 ---
PRESIDESY
REPUBLIIT INDONESIA
-4-
4
(71 Simbol o sebagaimana dimaksud pada ayat (6)
ditentukan nilainya oleh dewan pengupahan provinsi
atau dewan pengupahan kabupatenlkota dengan
mempertimbangkan:
a. tingkat penyerapan tenaga kerja; dan
b. rata-rata atau median Upah.
(S) Selain pertimbangan sebagaimana dimaksud
pada ayat (7), dalam menentukan o dapat
mempertimbangkan faktor lain yang relevan dengan
kondisi ketenagakerj aan.
(9) Jika nilai penyesuaian Upah minimum sebagaimana
dimaksud pada ayat (5) lebih kecil atau sama dengan
O (nol), Upah minimum yang akan ditetapkan sama
dengan nilai Upah minimum tahun berjalan.
(10) Data yang digunakan untuk penghitungan nilai
penyesuaian Upah minimum sebagaimana dimaksud
pada ayat (5) bersumber dari lembaga yang
berwenang di bidang statistik.
Di antara Pasal 26 dan Pasal 27 disisipkan 2 (dua) pasal,
yakni Pasal 26A dan Pasal 268 sehingga berbunyi sebagai
berikut:
Pasal 26A
(1) Dalam hal nilai Upah minimum tahun berjalan pada
wilay

In [15]:
# Download seluruh repo ke folder lokal
local_model_path = snapshot_download(
    repo_id = "rizkyayub/rizbuy-submission-qwen3-indonesian",
    token   = os.getenv("HF_TOKEN")
)
print(f"Model tersimpan di: {local_model_path}")

# Load dari path lokal hasil download
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(
    local_model_path,
    fix_mistral_regex=True,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Model tersimpan di: C:\Users\Rizky\.cache\huggingface\hub\models--rizkyayub--rizbuy-submission-qwen3-indonesian\snapshots\cb7ef056ddcaf6258a6a16391ddbe89cf8566d4c


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

c:\Users\Rizky\OneDrive\Documents\Python\finetuning\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [16]:
text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=1000,
)
 
llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

Device set to use cuda:0


In [17]:
template = template = """<|im_start|>system
Anda adalah asisten AI ahli hukum ketenagakerjaan Indonesia yang bertugas membantu pengguna.
Gunakan hanya informasi yang tersedia pada konteks berikut untuk menjawab pertanyaan.
Jika jawaban tidak ditemukan dalam konteks tersebut, sampaikan dengan jujur bahwa Anda tidak mengetahui jawabannya.
ATURAN SANGAT PENTING: Anda HARUS melakukan seluruh proses berpikir (reasoning) di dalam tag <think> secara eksklusif menggunakan Bahasa Indonesia yang baku dan formal. Jawaban akhir Anda juga harus dalam Bahasa Indonesia.

Context:
{context}<|im_end|>
<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
"""
prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

In [18]:
rag_chain = (
    {"context": compression_retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [19]:
def ask_question(query):
    print(f"\n==================================================")
    print(f"Pertanyaan: {query}")
    print(f"==================================================")

    
    final_docs = compression_retriever.invoke(query)
    
    context_text = "\n\n".join([doc.page_content for doc in final_docs])
    
    response = rag_chain.invoke(query)

    print("Jawaban:")
    print(response)

    print("\nSumber Referensi:")
    for i, doc in enumerate(final_docs, start=1):
        sumber = doc.metadata.get('source', 'Tidak diketahui')
        print(f"{i}. File: {sumber} | Potongan Teks:")
        print(f"   \"{doc.page_content[:1000]}...\"") 

query = "Jika karyawan PKWT bekerja selama 18 bulan dengan gaji pokok Rp5 juta, berapa uang kompensasinya?"
ask_question(query)


Pertanyaan: Jika karyawan PKWT bekerja selama 18 bulan dengan gaji pokok Rp5 juta, berapa uang kompensasinya?


c:\Users\Rizky\OneDrive\Documents\Python\finetuning\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Jawaban:
<think>
Okay, let's see. The user is asking about the compensation for a Karyawan Paksa Waktu Tenaga Kerja (PKWT) who has worked for 18 months with a base salary of Rp5 million. I need to calculate the compensation based on the Indonesian Labor Law provisions mentioned in the provided documents.

First, I'll check the relevant sections from the documents. From the first document, under Pasal 16, it states that for PKWT working more than 12 months, the compensation is calculated proportionally. Specifically, if they work more than 12 months, the calculation is masa kerja x 1 bulan upah divided by 12. Wait, no, actually looking back at the text: "PKWT selama lebih dari 12 (dua belas) bulan, dihitung secara proporsional dengan perhitungan: masa kerja x 1 (satu) bulan Upah." So, for each full year, they get one month's salary. But since the worker has been employed for 18 months, which is 1.5 years, then the compensation would be 1.5 months' worth of salary. 

Wait, but the exact 

In [ ]:
def jawab_pertanyaan(query):
    # Melakukan inference / generation
    response = rag_chain.invoke(query)
    return response

demo = gr.Interface(
    fn=jawab_pertanyaan,
    inputs=gr.Textbox(lines=2, placeholder="Ketik pertanyaan Anda tentang aturan ketenagakerjaan di sini..."),
    outputs=gr.Markdown(), # Menggunakan output Markdown agar rapi
    title="RAG Asisten Ketenagakerjaan (PP 35/2021 & PP 51/2023)",
    description="Sistem AI yang di-finetune untuk menjawab pertanyaan berdasarkan dokumen PDF regulasi ketenagakerjaan."
)

gc.collect()
torch.cuda.empty_cache()

# Coba jalankan ini di cell baru, jangan lewat Gradio dulu
test_response = rag_chain.invoke("Apa itu upah minimum?")
print(test_response)

# Jalankan Gradio
# Tambahkan debug=True
demo.launch(share=True) # share=True agar kamu dapat link publik yang bisa diakses

c:\Users\Rizky\OneDrive\Documents\Python\finetuning\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


<think>
Okay, the user is asking what the minimum wage is. Let me check the provided documents to find the answer.

Looking at the first document, it mentions that the minimum wage is determined based on economic conditions and labor force. It also states that the President of the Republic of Indonesia has the authority to set the minimum wage for provinces, cities, or districts. The documents also talk about adjustments to the minimum wage every year considering factors like economic growth, inflation, and an index related to workers' contributions to economic growth. There's a formula mentioned for calculating the adjusted minimum wage, which includes variables for inflation and a coefficient representing the contribution of labor to economic growth. 

In the second document, there's a reference to a regulation that amended the previous one, allowing for higher wages for employees with less than a year of service if they meet certain qualifications. Also, the minimum wage applies to 

c:\Users\Rizky\OneDrive\Documents\Python\finetuning\.venv\Lib\site-packages\gradio\routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Rizky\OneDrive\Documents\Python\finetuning\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\Rizky\OneDrive\Documents\Python\finetuning\.venv\Lib\site-packages\gradio\routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Rizky\OneDrive\Documents\Python\finetuning\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:464: FutureWarning: _check_is_size will be r